In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### PART 1-2: IMPORTS, CONFIGURATION, AND PATH MANAGEMENT

In [ ]:
# -*- coding: utf-8 -*-
"""FWI_hiding_system_v7.0 (Global Best Victim)"""

# =============================================================================
# PART 1: IMPORTS, CONFIGURATION, AND PATH MANAGEMENT
# =============================================================================

import math
import copy
import time
import random
import json
import os
import sys
import heapq
import shutil
import logging
import traceback
from logging.handlers import RotatingFileHandler
import collections
from collections import defaultdict, deque
from itertools import combinations
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Set, Optional, Deque
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import gc

# =============================================================================
# PART 2: GLOBAL CONFIGURATION AND PATH MANAGEMENT
# =============================================================================

@dataclass
class OptimizedConfig:
    # Thiết lập giới hạn an toàn cho Runtime 51GB RAM (Chừa 6GB cho OS)
    MAX_MEMORY_USAGE_MB: int = 45000
    BATCH_SIZE: int = 5000
    GC_FREQUENCY: int = 200 # Dọn rác thường xuyên hơn
    MAX_PATTERN_LENGTH: int = 7
    USE_MULTIPROCESSING: bool = True
    # Giảm số worker để tiết kiệm RAM khi chạy đa luồng
    NUM_WORKERS: int = min(4, max(1, mp.cpu_count() - 2))
    PROGRESS_REPORT_INTERVAL: int = 10000

config = OptimizedConfig()

@dataclass
class HidingConfig:
    seed: int = 42
    item_weights: Dict[str, float] = None

class PathManager:
    def __init__(self):
        self.is_colab = 'google.colab' in sys.modules
        if self.is_colab:
            self.DRIVE_ROOT = '/content/drive/MyDrive/HUTECH/Master/Master_Thesis/Sourcecode'
        else:
            self.DRIVE_ROOT = './'

        self.DATA_PATH = os.path.join(self.DRIVE_ROOT, 'Datasets/fwi_processed_datasets')
        self.LOGS_PATH = os.path.join(self.DRIVE_ROOT, 'MyLogs/fwi_processed_logs')
        self.RESULTS_PATH = os.path.join(self.DRIVE_ROOT, 'MyResults/fwi_processed_results')

        os.makedirs(self.LOGS_PATH, exist_ok=True)
        os.makedirs(self.RESULTS_PATH, exist_ok=True)
        os.makedirs(self.DATA_PATH, exist_ok=True)

    def get_data_path(self, filename: str) -> str:
        return os.path.join(self.DATA_PATH, filename)

    def get_log_path(self, filename: str) -> str:
        return os.path.join(self.LOGS_PATH, filename)

    def get_results_path(self, filename: str) -> str:
        return os.path.join(self.RESULTS_PATH, filename)

def setup_logging(log_file_path: str):
    root_logger = logging.getLogger()
    for handler in root_logger.handlers[:]:
        root_logger.removeHandler(handler)
        handler.close()
    root_logger.setLevel(logging.INFO)
    file_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    console_formatter = logging.Formatter('%(message)s')

    file_handler = logging.FileHandler(log_file_path, mode='w', encoding='utf-8')
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(file_formatter)
    root_logger.addHandler(file_handler)

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(console_formatter)
    root_logger.addHandler(console_handler)



### PART 3: CORE fwi MINING ENGINE (ORIGINAL VERSION) - DO NOT MODIFY

In [ ]:
# =============================================================================
# PART 3: CORE fwi MINING ENGINE (ORIGINAL VERSION) - DO NOT MODIFY
# =============================================================================
class OptimizedWUNNode_v1_Mock:
    def __init__(self, itemset, ws, tids):
        self.itemset = itemset
        self.ws = ws
        self.tids = tids

class MemoryMonitor:
    def __init__(self, max_memory_mb):
        self.max_memory_mb = max_memory_mb
        self.process = psutil.Process(os.getpid())
    def check_memory_usage(self):
        memory_mb = self.process.memory_info().rss / (1024 * 1024)
        if memory_mb > self.max_memory_mb:
            gc.collect()
            if psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024) > self.max_memory_mb:
                raise MemoryError(f"Memory usage ({memory_mb:.1f}MB) exceeds limit ({self.max_memory_mb}MB)")
        return memory_mb

class OptimizedWUNNode_v1:
    __slots__ = ['item_name', 'children', 'pre', 'post', 'weight', 'parent', 'tids']
    def __init__(self, item_name, parent):
        self.item_name, self.parent = item_name, parent; self.children = {}; self.tids = set(); self.pre, self.post, self.weight = -1, -1, 0.0

class OptimizedWUNTree_v1:
    def __init__(self, item_weights):
        self.root = OptimizedWUNNode_v1(None, None); self.node_index = {}; self.item_weights = item_weights; self.node_count = 1; self.memory_monitor = MemoryMonitor(config.MAX_MEMORY_USAGE_MB)
    def insert_transaction_batch(self, transactions_batch: List[Dict], utilities_batch: List[float], tids_batch: List[str]):
        for i, (transaction, utility, tid) in enumerate(zip(transactions_batch, utilities_batch, tids_batch)):
            # Truyền tid (string) vào hàm insert
            self.insert_transaction(transaction, utility, tid)
            if (i + 1) % config.GC_FREQUENCY == 0:
                self.memory_monitor.check_memory_usage()
    def insert_transaction(self, transaction: Dict, utility: float, transaction_id: str): # Chỉ định rõ type
        current = self.root
        for item in transaction.keys():
            if item not in current.children:
                current.children[item] = OptimizedWUNNode_v1(item, current);
                self.node_count += 1
            current = current.children[item];
            current.weight += utility;
            current.tids.add(transaction_id) # Bây giờ transaction_id là "T1", "T2"...
    def build_node_index_optimized(self):
        pre_counter, post_counter = 0, 0
        stack_pre = [self.root]
        while stack_pre:
            node = stack_pre.pop()
            if node.pre == -1: node.pre = pre_counter; pre_counter += 1
            sorted_children = sorted(node.children.keys(), key=lambda x: self.item_weights.get(x, 0), reverse=False)
            for child_item in sorted_children: stack_pre.append(node.children[child_item])
        stack_post1, stack_post2 = [self.root], []
        while stack_post1:
            node = stack_post1.pop(); stack_post2.append(node)
            sorted_children = sorted(node.children.keys(), key=lambda x: self.item_weights.get(x, 0), reverse=True)
            for child_item in sorted_children: stack_post1.append(node.children[child_item])
        while stack_post2:
            node = stack_post2.pop()
            if node.post == -1: node.post = post_counter; post_counter += 1
        all_nodes_q = collections.deque([self.root])
        while all_nodes_q:
            node = all_nodes_q.popleft()
            if node.pre != -1 and node.item_name is not None: self.node_index[node.pre] = node.post
            all_nodes_q.extend(node.children.values())
        return self.node_index
    def get_swun_list_optimized(self, item):
        swun_list, queue = [], collections.deque([self.root])
        while queue:
            node = queue.popleft()
            if node.item_name == item: swun_list.append((node.pre, node.weight, node.tids))
            queue.extend(node.children.values())
        return sorted(swun_list, key=lambda x: x[0])
    def swunl_intersection_optimized(self, swunl_y, swunl_x, ws_y, ws_x, sumtw, min_ws):
        if not swunl_x or not swunl_y: return None
        result, sc, i, j = [], 0.0, 0, 0
        cr_y_val = sum(w for _, w, _ in swunl_y); cr_x_val = sum(w for _, w, _ in swunl_x)
        min_abs_utility = min_ws * sumtw
        if min(cr_y_val, cr_x_val) < min_abs_utility: return None
        while i < len(swunl_x) and j < len(swunl_y):
            id_x, weight_x, _ = swunl_x[i]; id_y, weight_y, tids_y = swunl_y[j]
            post_i = self.node_index.get(id_x, -1); post_j = self.node_index.get(id_y, -1)
            if id_y < id_x:
                if post_j > post_i:
                    if result and result[-1][0] == id_y: result[-1] = (id_y, result[-1][1] + weight_x, result[-1][2].union(tids_y))
                    else: result.append((id_y, weight_x, tids_y.copy()))
                    sc += weight_x; cr_x_val -= weight_x; i += 1
                else: cr_y_val -= weight_y; j += 1
            else: cr_x_val -= weight_x; i += 1
            if sc + min(cr_x_val, cr_y_val) < min_abs_utility: return None
        return result

def compute_tw_optimized(transactions: Dict[str, Dict[str, int]], weights: Dict[str, float]):
    """Tính toán tw, trả về map [tid -> tw]"""
    tw, sumtw = defaultdict(float), 0.0
    transaction_tw_map = {} # Sử dụng map (dict)
    for tid, t in transactions.items(): # Lặp qua items() để lấy cả tid
        s_tk = len(t)
        if s_tk > 0:
            utility = sum(weights.get(item, 0.0) * qty for item, qty in t.items())
            tw_t = utility / s_tk
            sumtw += tw_t
            transaction_tw_map[tid] = tw_t # Lưu tw bằng tid (string)
            for item in t:
                tw[item] += tw_t
        else:
            transaction_tw_map[tid] = 0.0 # Lưu tw bằng tid (string)
    return tw, sumtw, transaction_tw_map # Trả về map

def calculate_ws_from_swunl_fast(swunl, sumtw):
    if not swunl or sumtw == 0: return 0.0
    return sum(weight for _, weight, _ in swunl) / sumtw

class ProgressiveMinor:
    def __init__(self):
        self.patterns_found = 0; self.start_time = time.time()
    def should_continue(self, current_length):
        return current_length < config.MAX_PATTERN_LENGTH
    def report_progress(self):
        self.patterns_found += 1
        if self.patterns_found % config.PROGRESS_REPORT_INTERVAL == 0:
            logging.getLogger(__name__).debug(f"Progress: {self.patterns_found} patterns found.")

def find_fwi_optimized(Prefix, L_k, S_k, tree, sumtw, fwis, seen_patterns, item_order, min_ws, progressive_minor):
    if not progressive_minor.should_continue(len(Prefix)): return
    for i in range(len(L_k) - 1, -1, -1):
        item_i_obj = L_k[i]; Prefix_next = Prefix + item_i_obj['pattern']
        if len(Prefix_next) > config.MAX_PATTERN_LENGTH: continue
        L_next, S_next = [], list(S_k)
        for j in range(i):
            item_j_obj = L_k[j]
            new_swunl = tree.swunl_intersection_optimized(item_i_obj['swunl'], item_j_obj['swunl'], item_i_obj['ws'], item_j_obj['ws'], sumtw, min_ws)
            if new_swunl:
                new_ws = calculate_ws_from_swunl_fast(new_swunl, sumtw)
                if new_ws >= min_ws:
                    new_pattern = sorted(Prefix_next + item_j_obj['pattern']); new_pattern_key = tuple(new_pattern)
                    tids = set().union(*(t for _, _, t in new_swunl))
                    candidate_c = {'pattern': item_j_obj['pattern'], 'ws': new_ws, 'swunl': new_swunl, 'tids': tids}
                    if new_pattern_key not in seen_patterns:
                        fwis.append({'pattern': new_pattern, 'ws': new_ws, 'tids': tids}); seen_patterns.add(new_pattern_key)
                        progressive_minor.report_progress()
                    if abs(new_ws - item_i_obj['ws']) < 1e-9: S_next.append(item_j_obj)
                    else: L_next.append(candidate_c)
        if len(Prefix_next) < config.MAX_PATTERN_LENGTH:
            s_sorted_objs = sorted(S_next, key=lambda obj: item_order.get(obj['pattern'][0]))
            s_sorted_items = [s_obj['pattern'][0] for s_obj in s_sorted_objs]
            l_sorted_objs = sorted(L_next, key=lambda obj: item_order.get(obj['pattern'][0]))
            prefix_next_ws = item_i_obj['ws']
            prefix_next_tids = set().union(*(t for _, _, t in item_i_obj['swunl']))
            if s_sorted_items: find_fwi_same_ws_optimized(Prefix_next, s_sorted_items, prefix_next_ws, prefix_next_tids, fwis, seen_patterns, progressive_minor)
            if l_sorted_objs: find_fwi_optimized(Prefix_next, l_sorted_objs, s_sorted_objs, tree, sumtw, fwis, seen_patterns, item_order, min_ws, progressive_minor)

def find_fwi_same_ws_optimized(Prefix, S, prefix_ws, prefix_tids, fwis, seen_patterns, progressive_minor):
    for r in range(1, len(S) + 1):
        if len(Prefix) + r > config.MAX_PATTERN_LENGTH: continue
        for subset in combinations(S, r):
            new_pattern = sorted(Prefix + list(subset)); new_pattern_key = tuple(new_pattern)
            if new_pattern_key not in seen_patterns:
                fwis.append({'pattern': new_pattern, 'ws': prefix_ws, 'tids': prefix_tids}); seen_patterns.add(new_pattern_key)
                progressive_minor.report_progress()

g_tree, g_swunl_dict, g_ws_dict, g_sumtw, g_min_ws, g_item_order_map, g_item_order_list, g_base_fwis = (None,) * 8

def init_worker(tree, swunl_dict, ws_dict, sumtw, min_ws, item_order_map, item_order_list, base_fwis):
    global g_tree, g_swunl_dict, g_ws_dict, g_sumtw, g_min_ws, g_item_order_map, g_item_order_list, g_base_fwis
    g_tree, g_swunl_dict, g_ws_dict, g_sumtw, g_min_ws, g_item_order_map, g_item_order_list, g_base_fwis = \
        tree, swunl_dict, ws_dict, sumtw, min_ws, item_order_map, item_order_list, base_fwis

def mine_i1_chunk(i1_chunk_indices):
    progressive_minor = ProgressiveMinor()
    fwis = list(g_base_fwis); seen_patterns = {tuple(p['pattern']) for p in fwis}
    initial_fwis_count = len(fwis)
    for item_x_index in i1_chunk_indices:
        item_x = g_item_order_list[item_x_index]
        if not progressive_minor.should_continue(1): break
        Prefix, L, S = [item_x], [], []; ws_x = g_ws_dict.get(item_x, 0)
        for y_index in range(item_x_index):
            item_y = g_item_order_list[y_index]
            swunl_xy = g_tree.swunl_intersection_optimized(g_swunl_dict.get(item_y), g_swunl_dict.get(item_x), g_ws_dict.get(item_y, 0.0), ws_x, g_sumtw, g_min_ws)
            if swunl_xy:
                ws_xy_val = calculate_ws_from_swunl_fast(swunl_xy, g_sumtw)
                if ws_xy_val >= g_min_ws:
                    new_pattern_key = tuple(sorted([item_x, item_y]))
                    if new_pattern_key not in seen_patterns:
                        tids_xy = set().union(*(t for _, _, t in swunl_xy))
                        fwis.append({'pattern': list(new_pattern_key), 'ws': ws_xy_val, 'tids': tids_xy}); seen_patterns.add(new_pattern_key)
                    item_y_obj = {'pattern': [item_y], 'ws': ws_xy_val, 'swunl': swunl_xy}
                    if abs(ws_xy_val - ws_x) < 1e-9: S.append(item_y_obj)
                    else: L.append(item_y_obj)
        s_sorted_objs = sorted(S, key=lambda obj: g_item_order_map.get(obj['pattern'][0]))
        s_sorted_items = [s_obj['pattern'][0] for s_obj in s_sorted_objs]
        l_sorted_objs = sorted(L, key=lambda obj: g_item_order_map.get(obj['pattern'][0]))
        item_x_tids = set().union(*(t for _, _, t in g_swunl_dict.get(item_x,[])))
        if s_sorted_items: find_fwi_same_ws_optimized(Prefix, s_sorted_items, ws_x, item_x_tids, fwis, seen_patterns, progressive_minor)
        if l_sorted_objs: find_fwi_optimized(Prefix, l_sorted_objs, s_sorted_objs, g_tree, g_sumtw, fwis, seen_patterns, g_item_order_map, g_min_ws, progressive_minor)
    return fwis[initial_fwis_count:]

def run_fwi_mining_core(transactions: Dict[str, Dict[str, int]], item_weights: Dict[str, float], min_ws: float) -> List[Dict[str, Any]]:
    logger = logging.getLogger(__name__)
    logger.info(f"Bắt đầu khai phá fwis (phiên bản gốc v5.1) với min_ws = {min_ws:.6f}")
    if not transactions or not item_weights: return []

    # KHỐI MÃ MỚI
    tw, sumtw, transaction_tw_map = compute_tw_optimized(transactions, item_weights) # Sửa 1
    if sumtw == 0: return []

    ws = {item: val / sumtw for item, val in tw.items()}
    I1 = sorted([item for item, w in ws.items() if w >= min_ws], key=ws.get, reverse=True)
    logger.info(f"I1: {len(I1)}") # Sửa 2 (dùng logger)
    item_order_map = {item: i for i, item in enumerate(I1)}

    # Sửa 3: Xử lý giao dịch và giữ lại TIDs
    processed_transactions_map = {}
    for tid, t in transactions.items():
        processed_t = dict(sorted({item: qty for item, qty in t.items() if item in item_order_map}.items(), key=lambda x: item_order_map.get(x[0])))
        if processed_t: # Chỉ thêm nếu giao dịch không rỗng
            processed_transactions_map[tid] = processed_t

    # [QUAN TRỌNG] Giải phóng memory ngay lập tức
    # del transactions
    # gc.collect()

    tree = OptimizedWUNTree_v1(item_weights)

    # Sửa 4: Chuẩn bị dữ liệu để đưa TIDs (string) vào cây
    tids_list = list(processed_transactions_map.keys())
    transactions_list = [processed_transactions_map[tid] for tid in tids_list]
    tw_list = [transaction_tw_map[tid] for tid in tids_list]

    # Gọi hàm batch insert đã được sửa đổi
    tree.insert_transaction_batch(transactions_list, tw_list, tids_list)
    tree.build_node_index_optimized()

    swunl_dict = {item: tree.get_swun_list_optimized(item) for item in I1}
    base_fwis = [{'pattern': [item], 'ws': ws[item], 'tids': set.union(*(t for _, _, t in swunl_dict.get(item,[])))} for item in I1]
    final_fwis = list(base_fwis)

    if config.USE_MULTIPROCESSING and config.NUM_WORKERS > 1 and len(I1) > 1:
        logger.info(f"Sử dụng {config.NUM_WORKERS} worker(s) để khai phá...")
        i1_indices = list(range(len(I1)))
        chunk_size = math.ceil(len(i1_indices) / config.NUM_WORKERS)
        chunks = [i1_indices[i:i + chunk_size] for i in range(0, len(i1_indices), chunk_size)]
        init_args = (tree, swunl_dict, ws, sumtw, min_ws, item_order_map, I1, base_fwis)
        with ProcessPoolExecutor(max_workers=config.NUM_WORKERS, initializer=init_worker, initargs=init_args) as executor:
            futures = [executor.submit(mine_i1_chunk, chunk) for chunk in chunks]
            for future in as_completed(futures):
                try:
                    chunk_results = future.result(); final_fwis.extend(chunk_results)
                except Exception as e: logger.error(f"A worker process failed: {e}\n{traceback.format_exc()}")
    else:
        logger.info("Chạy khai phá ở chế độ đơn luồng...")
        init_args = (tree, swunl_dict, ws, sumtw, min_ws, item_order_map, I1, base_fwis)
        init_worker(*init_args)
        single_chunk_results = mine_i1_chunk(list(range(len(I1))))
        final_fwis.extend(single_chunk_results)

    logger.info(f"Khai phá hoàn tất. Tìm thấy {len(final_fwis)} fwis.")
    # return [{'pattern': [str(i) for i in p['pattern']], 'ws': p['ws']} for p in final_fwis]
    # return [{'pattern': [str(i) for i in p['pattern']], 'ws': p['ws'], 'tids': p.get('tids', set())} for p in final_fwis]
    formatted_fwis = []
    for p in final_fwis:
        formatted_fwis.append(OptimizedWUNNode_v1_Mock(p['pattern'], p['ws'], p.get('tids', set())))

    return formatted_fwis, tree, transaction_tw_map

### PART 4: HIDING ALGORITHMS (HFPriorityManager vs. MCPriorityManager) WITH HEARTBEAT MONITOR


In [ ]:
# =============================================================================
# PART 4: HIDING ALGORITHMS (WITH HEARTBEAT MONITOR)
# =============================================================================

import heapq
import copy
import time
import logging

class HFPriorityManager:
    """ALGORITHM A: HFPriorityManager (The Hunter)"""
    def __init__(self, item_weights, transaction_weights_ref, timeout=3600, logger=None):
        self.item_weights = item_weights
        self.tw_ref = copy.deepcopy(transaction_weights_ref)
        self.logger = logger if logger else logging.getLogger(__name__)
        self.TIMEOUT = timeout
        self.name = "HFPriorityManager"
        self.current_tsw = sum(self.tw_ref.values()) or 1.0
        self.stats = {"deleted": 0, "status": "Completed"}

    def sanitize(self, sfwi_nodes, transactions, min_ws):
        start_time = time.time()
        targets = sorted(sfwi_nodes, key=lambda x: x.ws, reverse=True)

        total_targets = len(targets)
        self.logger.info(f"   [{self.name}] Started. Targets: {total_targets} patterns.")

        for i, s_node in enumerate(targets):
            # Check Timeout cấp độ Pattern
            if time.time() - start_time > self.TIMEOUT:
                self.stats["status"] = "Timeout"
                self.logger.warning(f"   ! [{self.name}] Timeout triggered at pattern {i}/{total_targets}.")
                break

            # [HEARTBEAT] Log tiến độ mỗi khi xử lý xong 1 pattern
            if i % 1 == 0:  # Log mỗi pattern để theo dõi sát sao hơn
                elapsed = time.time() - start_time
                self.logger.info(f"      > Processing pattern {i+1}/{total_targets} (Elapsed: {elapsed:.0f}s)...")

            # Vòng lặp xóa
            loop_count = 0
            while True:
                # Check Timeout cấp độ vi mô (tránh kẹt trong 1 pattern quá lâu)
                if time.time() - start_time > self.TIMEOUT:
                    self.stats["status"] = "Timeout"; break

                if self._calc_ws(s_node, transactions) < min_ws: break

                s_set = set(s_node.itemset)
                candidates = [tid for tid in s_node.tids if s_set.issubset(transactions[tid])]

                if not candidates:
                    self.logger.warning(f"      ! Pattern {s_node.itemset} stuck (No candidates). Skipping.")
                    break

                candidates.sort(key=lambda t: self.tw_ref[t], reverse=True)

                hit = False
                for tid in candidates:
                    victim_pool = [i for i in transactions[tid] if i in s_set]
                    if not victim_pool: continue
                    victim = max(victim_pool, key=lambda i: self.item_weights.get(i, 0))

                    self._delete_item(tid, victim, transactions)
                    self.stats["deleted"] += 1
                    hit = True
                    loop_count += 1

                    # [HEARTBEAT] Log mỗi 100 lần xóa
                    if self.stats["deleted"] % 100 == 0:
                         self.logger.info(f"         ...Deleted {self.stats['deleted']} items total...")
                    break

                if not hit: break

        return {"Runtime": time.time() - start_time, "Stats": self.stats}

    def _calc_ws(self, node, trans):
        num = sum(self.tw_ref[tid] for tid in node.tids if set(node.itemset).issubset(trans[tid]))
        return num / self.current_tsw

    def _delete_item(self, tid, item, trans):
        t = trans[tid]; t.remove(item)
        old_tw = self.tw_ref[tid]
        new_tw = sum(self.item_weights.get(i, 0) for i in t)/len(t) if t else 0.0
        self.tw_ref[tid] = new_tw
        self.current_tsw -= (old_tw - new_tw)


class MCPriorityManager:
    """ALGORITHM B: MCPriorityManager (The MCPriorityManager)"""
    def __init__(self, item_weights, transaction_weights_ref, timeout=3600, logger=None):
        self.item_weights = item_weights
        self.tw_ref = copy.deepcopy(transaction_weights_ref)
        self.logger = logger if logger else logging.getLogger(__name__)
        self.TIMEOUT = timeout
        self.name = "MCPriorityManager"
        self.current_tsw = sum(self.tw_ref.values()) or 1.0
        self.nsfwi_map = defaultdict(list)
        self.stats = {"deleted": 0, "vetoed": 0, "status": "Completed"}

    def sanitize(self, sfwi_nodes, nsfwi_nodes, transactions, min_ws):
        start_time = time.time()
        self.logger.info(f"      [{self.name}] Building Safety Net (NSFWI Index)...")
        # Build index nhanh hơn
        for node in nsfwi_nodes:
            for tid in node.tids: self.nsfwi_map[tid].append(node)

        targets = sorted(sfwi_nodes, key=lambda x: x.ws, reverse=False)
        total_targets = len(targets)

        for i, s_node in enumerate(targets):
            if time.time() - start_time > self.TIMEOUT:
                self.stats["status"] = "Timeout"
                self.logger.warning(f"   ! [{self.name}] Timeout triggered at pattern {i}/{total_targets}.")
                break

            if i % 1 == 0:
                 elapsed = time.time() - start_time
                 self.logger.info(f"      > Processing pattern {i+1}/{total_targets} (Elapsed: {elapsed:.0f}s)...")

            while True:
                if time.time() - start_time > self.TIMEOUT:
                     self.stats["status"] = "Timeout"; break

                if self._calc_ws(s_node, transactions) < min_ws: break

                s_set = set(s_node.itemset)
                candidates = [tid for tid in s_node.tids if s_set.issubset(transactions[tid])]
                if not candidates: break

                candidates.sort(key=lambda t: self.tw_ref[t], reverse=False)

                move = False
                for tid in candidates:
                    pool = [i for i in transactions[tid] if i in s_set]
                    pool.sort(key=lambda i: self.item_weights.get(i, 0))

                    for item in pool:
                        if self._is_safe(tid, item, transactions[tid], min_ws):
                            self._delete_item(tid, item, transactions)
                            self.stats["deleted"] += 1
                            move = True
                            if self.stats["deleted"] % 100 == 0:
                                self.logger.info(f"         ...Deleted {self.stats['deleted']} items total...")
                            break
                        else: self.stats["vetoed"] += 1
                    if move: break
                if not move: break

        return {"Runtime": time.time() - start_time, "Stats": self.stats}

    def _calc_ws(self, node, trans):
        return sum(self.tw_ref[tid] for tid in node.tids if set(node.itemset).issubset(trans[tid])) / self.current_tsw

    def _is_safe(self, tid, item, current_items, min_ws):
        # (Logic giữ nguyên, chỉ thêm log heartbeat ở trên)
        old_tw = self.tw_ref[tid]
        rem = [i for i in current_items if i != item]
        new_tw = sum(self.item_weights.get(i, 0) for i in rem)/len(rem) if rem else 0.0
        delta = old_tw - new_tw
        pred_tsw = self.current_tsw - delta
        if pred_tsw <= 0: return False

        for node in self.nsfwi_map.get(tid, []):
            curr_sum = node.ws * self.current_tsw
            pred_sum = curr_sum - old_tw if item in node.itemset else curr_sum - delta
            if (pred_sum / pred_tsw) < min_ws: return False
        return True

    def _delete_item(self, tid, item, trans):
        t = trans[tid]; t.remove(item)
        old_tw = self.tw_ref[tid]
        new_tw = sum(self.item_weights.get(i, 0) for i in t)/len(t) if t else 0.0
        self.tw_ref[tid] = new_tw
        self.current_tsw -= (old_tw - new_tw)

### PART 6: MAIN EXECUTION (v38 - DUAL TRACK WITH CACHE)

In [ ]:
# =============================================================================
# PART 6: MAIN EXECUTION (v38 FINAL - DUAL TRACK WITH CACHE & HELPERS)
# =============================================================================

import pickle
import json
import os
import copy
import logging
import gc

# --- 1. HELPER FUNCTIONS (Đã bổ sung đầy đủ) ---

def load_transactions_from_file(filepath):
    """Đọc file transaction và trả về dict {tid: {item: qty}}"""
    if not os.path.exists(filepath): return {}
    transactions = {}
    try:
        with open(filepath, 'r') as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line: continue
                parts = line.split()
                items = {}
                for part in parts:
                    if ':' in part:
                        item, qty = part.split(':')
                        items[item] = int(qty)
                    else:
                        items[part] = 1
                transactions[f"T{i+1}"] = items
    except Exception as e:
        print(f"Error loading transactions: {e}")
    return transactions

def load_weights_from_file(filepath):
    """Đọc file weights và trả về dict {item: weight}"""
    if not os.path.exists(filepath): return {}
    weights = {}
    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.replace(',', ':').split(':')
                if len(parts) >= 2:
                    weights[parts[0]] = float(parts[1])
    except Exception as e:
        print(f"Error loading weights: {e}")
    return weights

class SimpleSFWISelector:
    """Class chọn lọc SFWI (Top-k)"""
    @staticmethod
    def select_top_k(fwi_list, k=50):
        # Sắp xếp theo ws giảm dần và lấy top k
        sorted_fwi = sorted(fwi_list, key=lambda x: x.ws, reverse=True)
        return sorted_fwi[:k]

def evaluate_comprehensive(original_fwis, sanitized_fwis, sfwi_patterns, nsfwi_patterns,
                           original_trans, sanitized_trans, original_tw, sanitized_tw, runtime):
    """Tính toán đầy đủ 8 chỉ số đo lường."""
    # Chuyển đổi list object sang set of tuples để so sánh
    set_orig = {tuple(p.itemset) for p in original_fwis}
    set_new = {tuple(p.itemset) for p in sanitized_fwis}
    set_sfwi = {tuple(p.itemset) for p in sfwi_patterns}
    set_nsfwi = {tuple(p.itemset) for p in nsfwi_patterns}

    # 1. HF
    rem_sfwi = set_sfwi.intersection(set_new)
    hf = (len(rem_sfwi) / len(set_sfwi)) * 100 if set_sfwi else 0

    # 2. MC
    lost_nsfwi = set_nsfwi - set_new
    mc = (len(lost_nsfwi) / len(set_nsfwi)) * 100 if set_nsfwi else 0

    # 3. AC
    artificial = set_new - set_orig
    ac = (len(artificial) / len(set_new)) * 100 if set_new else 0

    # 4. IUS
    dict_orig_ws = {tuple(p.itemset): p.ws for p in original_fwis}
    dict_new_ws = {tuple(p.itemset): p.ws for p in sanitized_fwis}
    total_util_orig = sum(dict_orig_ws.values())
    total_util_new = sum(dict_new_ws.values())
    ius = (total_util_new / total_util_orig) * 100 if total_util_orig > 0 else 0

    # 5. DUS
    dus = (sum(sanitized_tw.values()) / sum(original_tw.values()) * 100) if original_tw else 0

    # 6. TMR
    mod = sum(1 for t in original_trans if len(original_trans[t]) != len(sanitized_trans[t]))
    tmr = mod / len(original_trans) * 100

    # 7. DDI
    o_it = sum(len(t) for t in original_trans.values())
    n_it = sum(len(t) for t in sanitized_trans.values())
    ddi = (o_it - n_it) / o_it * 100 if o_it else 0

    return {
        "HF": hf, "MC": mc, "AC": ac,
        "IUS": ius, "DUS": dus,
        "TMR": tmr, "DDI": ddi, "Runtime": runtime
    }

# --- 2. MAIN EXPERIMENT RUNNER (Dual-Track) ---

def run_dual_experiment_with_cache():
    path_mgr = PathManager()
    setup_logging(path_mgr.get_log_path('log_v38_dual_algo.txt'))
    logger = logging.getLogger(__name__)

    # 1. Cấu hình 7 Tập dữ liệu
    datasets = {
        "Retail": ("retail_quantities.txt", "retail_weights.txt", 0.01),
        "BMS-POS": ("bms-pos_quantities.txt", "bms-pos_weights.txt", 0.001),
        "Chainstore": ("chainstore_quantities.txt", "chainstore_weights.txt", 0.007),
        "Kosarak": ("kosarak_quantities.txt", "kosarak_weights.txt", 0.015),
        "Mushroom": ("mushroom_quantities.txt", "mushroom_weights.txt", 0.07),
        "Accidents": ("accident_quantities.txt", "accident_weights.txt", 0.6),
        "Chess": ("chess_fimi_quantities.txt", "chess_fimi_weights.txt", 0.5)
    }

    res_file = path_mgr.get_results_path('fwi_v38_dual_results.json')
    all_results = {}
    if os.path.exists(res_file):
        try: all_results = json.load(open(res_file))
        except: pass

    # Định nghĩa 2 Thuật toán
    algorithms = [
        ("HFPriorityManager", HFPriorityManager),
        ("MCPriorityManager", MCPriorityManager)
    ]

    TIMEOUT = 3600 # 60 phút cho mỗi thuật toán

    # 2. Vòng lặp qua từng Dataset
    for d_name, (t_file, w_file, min_ws) in datasets.items():
        if d_name not in all_results: all_results[d_name] = {}

        # Check completion
        if "HFPriorityManager" in all_results[d_name] and "MCPriorityManager" in all_results[d_name]:
            logger.info(f"Skipping {d_name} (Both algorithms completed).")
            continue

        logger.info(f"==========================================")
        logger.info(f"--- PREPARING DATASET: {d_name} ---")

        try:
            # 2.1 Load Data
            trans_D = load_transactions_from_file(path_mgr.get_data_path(t_file))
            w = load_weights_from_file(path_mgr.get_data_path(w_file))

            # 2.2 Mining Original with Smart Cache
            cache_path = path_mgr.get_data_path(f"{d_name}_mining_cache.pkl")
            fwi_orig, tw_orig = None, None

            # Try loading cache
            if os.path.exists(cache_path):
                logger.info("   > Loading Mining Cache...")
                try:
                    with open(cache_path, 'rb') as f:
                        cache_data = pickle.load(f)
                        fwi_orig = cache_data['fwi']
                        tw_orig = cache_data['tw_map']
                    logger.info(f"   > Loaded {len(fwi_orig)} FWIs from cache.")
                except Exception as e:
                    logger.warning(f"   ! Cache corrupt: {e}. Re-mining.")

            # If no cache, mine
            if fwi_orig is None:
                logger.info("   > Mining Original D...")
                fwi_orig, _, tw_orig = run_fwi_mining_core(trans_D, w, min_ws)
                if fwi_orig:
                    logger.info("   > Saving Cache...")
                    with open(cache_path, 'wb') as f:
                        pickle.dump({'fwi': fwi_orig, 'tw_map': tw_orig}, f)

            if not fwi_orig:
                logger.error(f"   ! No FWI found for {d_name}")
                continue

            # Select Targets (Top 50)
            sfwi = SimpleSFWISelector.select_top_k(fwi_orig, k=50)
            sfwi_ids = {id(n) for n in sfwi}
            nsfwi = [n for n in fwi_orig if id(n) not in sfwi_ids]

            # Template Set-based Transaction (để deepcopy nhanh hơn)
            trans_set_template = {k: set(v.keys()) for k,v in trans_D.items()}

            # 3. Chạy Song Mã (HFPriorityManager & MCPriorityManager)
            for algo_name, AlgoClass in algorithms:
                if algo_name in all_results[d_name]:
                    logger.info(f"Skipping {algo_name} for {d_name} (Done).")
                    continue

                logger.info(f">>> Running {algo_name} on {d_name}...")

                try:
                    # 3.1 Khởi tạo D' riêng cho thuật toán
                    if algo_name == "HFPriorityManager":
                        trans_HFPriorityManager = copy.deepcopy(trans_set_template)
                        t_run = trans_HFPriorityManager
                    else:
                        trans_MCPriorityManager = copy.deepcopy(trans_set_template)
                        t_run = trans_MCPriorityManager

                    # 3.2 Khởi tạo Manager & Run
                    manager = AlgoClass(w, tw_orig, TIMEOUT, logger)

                    if algo_name == "MCPriorityManager":
                        res_int = manager.sanitize(sfwi, nsfwi, t_run, min_ws)
                    else:
                        res_int = manager.sanitize(sfwi, t_run, min_ws)

                    # 3.3 Re-mine (Evaluation on D')
                    logger.info("   > Re-mining sanitized dataset...")
                    remine_trans = {tid: {k: trans_D[tid][k] for k in items} for tid, items in t_run.items()}
                    fwi_new, _, tw_new = run_fwi_mining_core(remine_trans, w, min_ws)

                    # 3.4 Metrics
                    metrics = evaluate_comprehensive(
                        original_fwis=fwi_orig, sanitized_fwis=fwi_new,
                        sfwi_patterns=sfwi, nsfwi_patterns=nsfwi,
                        original_trans=trans_D, sanitized_trans=remine_trans,
                        original_tw=tw_orig, sanitized_tw=tw_new,
                        runtime=res_int['Runtime']
                    )
                    metrics['InternalStats'] = res_int['Stats']

                    logger.info(f"   [{algo_name}] HF: {metrics['HF']:.1f}% | MC: {metrics['MC']:.2f}%")

                    # Save
                    all_results[d_name][algo_name] = metrics
                    with open(res_file, 'w') as f: json.dump(all_results, f, indent=4)

                except MemoryError:
                    logger.error(f"   [CRASH] Memory Limit Exceeded ({algo_name}).")
                    all_results[d_name][algo_name] = {"Error": "Memory Limit Exceeded"}
                    with open(res_file, 'w') as f: json.dump(all_results, f, indent=4)
                    gc.collect()
                except Exception as e:
                    logger.error(f"   [CRASH] Error ({algo_name}): {e}")
                    all_results[d_name][algo_name] = {"Error": str(e)}
                    with open(res_file, 'w') as f: json.dump(all_results, f, indent=4)
                    gc.collect()

                # Clean up algo memory
                try: del t_run, remine_trans, manager
                except: pass
                gc.collect()

        except Exception as e_dataset:
            logger.error(f"!!! CRITICAL FAILURE ON DATASET {d_name}: {e_dataset}")

        finally:
            # Clean up dataset memory
            logger.info("Cleaning up memory...")
            try: del trans_D, w, fwi_orig, tw_orig, sfwi, nsfwi, trans_set_template
            except: pass
            gc.collect()

if __name__ == "__main__":
    run_dual_experiment_with_cache()

Skipping Retail (Both algorithms completed).
Skipping BMS-POS (Both algorithms completed).
Skipping Chainstore (Both algorithms completed).
Skipping Kosarak (Both algorithms completed).
Skipping Mushroom (Both algorithms completed).
Skipping Accidents (Both algorithms completed).
Skipping Chess (Both algorithms completed).


### **CÔNG CỤ TRỰC QUAN HÓA (VISUALIZATION CODE OPTIMIZED)**

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# ==============================================================================
# 1. CẤU HÌNH VÀ LOAD DỮ LIỆU
# ==============================================================================
file_path = '/content/drive/MyDrive/HUTECH/Master/Master_Thesis/Sourcecode/MyResults/fwi_processed_results/fwi_v38_dual_results.json'

# Định nghĩa màu sắc nhất quán: Đỏ = Hunter (HFPriority), Xanh = Guardian (MCPriority)
COLOR_PALETTE = {
    'HFPriority': '#d62728',
    'MCPriority': '#1f77b4'
}

try:
    with open(file_path, 'r') as f:
        data = json.load(f)

    # Chuyển đổi JSON sang DataFrame
    records = []
    for dataset, algos in data.items():
        for algo, metrics in algos.items():
            if "Error" in metrics: continue
            row = metrics.copy()
            row['Dataset'] = dataset
            row['Algorithm'] = 'HFPriority' if 'HF' in algo else 'MCPriority'
            records.append(row)

    df = pd.DataFrame(records)

    # Sắp xếp thứ tự Dataset: Đặc -> Thưa -> Lớn
    order = ['Mushroom', 'Accidents', 'Chess', 'Retail', 'Chainstore', 'BMS-POS', 'Kosarak']
    existing_order = [d for d in order if d in df['Dataset'].unique()]
    df['Dataset'] = pd.Categorical(df['Dataset'], categories=existing_order, ordered=True)
    df = df.sort_values('Dataset')

    print(">>> Dữ liệu đã được load và xử lý thành công!\n")

    # ==============================================================================
    # 2. HÀM VẼ BIỂU ĐỒ SO SÁNH CHỈ SỐ TRÊN TOÀN BỘ DATASET (Cho Fig 1, 2, 6, 7)
    # ==============================================================================
    def plot_metric_bar_chart(metric, ylabel, filename, is_runtime=False):
        plt.figure(figsize=(12, 6), dpi=300) # dpi=300 giúp ảnh cực nét
        sns.set_style("whitegrid")

        ax = sns.barplot(
            data=df, x='Dataset', y=metric, hue='Algorithm', palette=COLOR_PALETTE
        )

        plt.ylabel(ylabel, fontsize=13, fontweight='bold')
        plt.xlabel("Datasets", fontsize=13, fontweight='bold')
        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)

        # Đưa Legend lên vị trí của Title cũ (phía trên cùng)
        plt.legend(title=None, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=12, frameon=False)

        # Format trục Y và Label
        if not is_runtime:
            plt.ylim(0, 115) # Dành khoảng trống cho label %
            for container in ax.containers:
                ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=10)
        else:
            # Runtime không giới hạn 100, format số nguyên
            max_val = df[metric].max()
            plt.ylim(0, max_val * 1.15)
            for container in ax.containers:
                ax.bar_label(container, fmt='%.0f', padding=3, fontsize=10)

        plt.tight_layout()
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Đã lưu: {filename}")
        plt.close() # Đóng plot để giải phóng bộ nhớ

    # ==============================================================================
    # 3. HÀM VẼ BIỂU ĐỒ TRADE-OFF CHO TỪNG DATASET (Cho Fig 3, 4, 5)
    # ==============================================================================
    def plot_dataset_bar_chart(dataset_name, filename):
        subset = df[df['Dataset'] == dataset_name].copy()
        if subset.empty: return

        metrics = ['HF', 'MC', 'AC', 'DDI', 'TMR']
        df_melt = subset.melt(id_vars=['Algorithm'], value_vars=metrics,
                              var_name='Metric', value_name='Value (%)')

        plt.figure(figsize=(10, 6), dpi=300)
        sns.set_style("whitegrid")

        ax = sns.barplot(
            data=df_melt, x='Metric', y='Value (%)', hue='Algorithm', palette=COLOR_PALETTE
        )

        plt.ylim(0, 115)
        plt.ylabel("Percentage (%)", fontsize=13, fontweight='bold')
        plt.xlabel("Metrics", fontsize=13, fontweight='bold')
        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)

        # Đưa Legend lên vị trí của Title cũ (phía trên cùng)
        plt.legend(title=None, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=12, frameon=False)

        for container in ax.containers:
            ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=11)

        plt.tight_layout()
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"Đã lưu: {filename}")
        plt.close()

    # ==============================================================================
    # 4. THỰC HIỆN VẼ TOÀN BỘ BIỂU ĐỒ
    # ==============================================================================
    print("--- ĐANG TẠO HÌNH ẢNH CHO BÀI BÁO ---")

    # Nhóm 1: Privacy & Utility
    plot_metric_bar_chart('HF', 'Hiding Failure (%)', 'Fig_1_HF_Comparison.png')
    plot_metric_bar_chart('MC', 'Misses Cost (%)', 'Fig_2_MC_Comparison.png')

    # Nhóm 2: Trade-off Profiles (Thay thế Line chart cũ)
    plot_dataset_bar_chart('Chess', 'Fig_3_Chess_Tradeoff.png')
    plot_dataset_bar_chart('Retail', 'Fig_4_Retail_Tradeoff.png')
    plot_dataset_bar_chart('BMS-POS', 'Fig_5_BMS_POS_Tradeoff.png')

    # Nhóm 3: Side-effects & Performance
    plot_metric_bar_chart('AC', 'Artificial Cost (%)', 'Fig_6_AC_Comparison.png')
    plot_metric_bar_chart('IUS', 'IWS - Invalidated Weight Score (%)', 'Fig_8_IWS_Comparison.png')
    plot_metric_bar_chart('Runtime', 'Time (seconds)', 'Fig_7_Runtime_Comparison.png', is_runtime=True)

    print("\n>>> HOÀN TẤT! Tất cả hình ảnh đã được lưu ở độ phân giải cao (300 DPI).")

except FileNotFoundError:
    print(f"LỖI: Không tìm thấy file tại {file_path}")
except Exception as e:
    print(f"LỖI KHÔNG XÁC ĐỊNH: {e}")

>>> Dữ liệu đã được load và xử lý thành công!

--- ĐANG TẠO HÌNH ẢNH CHO BÀI BÁO ---
Đã lưu: Fig_1_HF_Comparison.png
Đã lưu: Fig_2_MC_Comparison.png
Đã lưu: Fig_3_Chess_Tradeoff.png
Đã lưu: Fig_4_Retail_Tradeoff.png
Đã lưu: Fig_5_BMS_POS_Tradeoff.png
Đã lưu: Fig_6_AC_Comparison.png
Đã lưu: Fig_8_IWS_Comparison.png
Đã lưu: Fig_7_Runtime_Comparison.png

>>> HOÀN TẤT! Tất cả hình ảnh đã được lưu ở độ phân giải cao (300 DPI).
